# BioKB (Neo4j) — Wound Healing & Mediterranean Plants Query Notebook

This notebook contains **ready-to-run Cypher queries** for BioKB Neo4j database focused on:



Note:
- the graph shows `(:Disease)` is connected via only two relationship types: `USED_FOR` and `MENTIONS_DISEASE`.
.


## 0) Setup: Connect to Neo4j


In [3]:
from neo4j import GraphDatabase

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "neo4j_password")

def query_neo4j(query, parameters=None):
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            result = session.run(query, parameters or {})
            return [dict(record) for record in result]

## 1) Confirm core schema pieces (optional but useful)

In [4]:
# List all labels
import pandas as pd

query = """
CALL db.labels()
"""
df_labels = pd.DataFrame(query_neo4j(query))
print("Total labels:", len(df_labels))
df_labels

Total labels: 71


,label
0,Journal
1,Article
2,Table
3,Taxon
4,Disease
...,...
66,Cofactor
67,Reaction
68,NSPReaction
69,EnzymeClass


In [21]:
# List all relationship types

query = """
CALL db.relationshipTypes()
"""
df_rels = pd.DataFrame(query_neo4j(query))
print("Total relationship types:", len(df_rels))
df_rels

Total relationship types: 42


,relationshipType
0,HAS_XREF
1,SAME_AS
2,HAS_NAME
3,IS_A
4,HAS_FUNCTIONAL_PARENT
5,HAS_ROLE
6,HAS_PART
7,HAS_PARENT_HYDRIDE
8,IS_SUBSTITUENT_GROUP_FROM
9,IS_CONJUGATE_BASE_OF


## 2) Disease layer: how Disease nodes connect

In [22]:
import pandas as pd

query = """
MATCH (:Disease)-[r]-()
RETURN type(r) AS rel_type,
       count(*) AS n
ORDER BY n DESC
"""

df = pd.DataFrame(query_neo4j(query))

print("Relationship types connected to Disease:", len(df))
df


Relationship types connected to Disease: 2


,rel_type,n
0,USED_FOR,31270
1,MENTIONS_DISEASE,16189


## 3) Wound-related Disease nodes (text search)

In [23]:

# TITLE: Find wound/burn/ulcer-related Disease entries (by diseaseName or name)
query = """
MATCH (d:Disease)
WHERE toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS 'wound'
   OR toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS 'burn'
   OR toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS 'ulcer'
RETURN coalesce(d.diseaseName, d.name) AS disease
LIMIT 20
"""
query_neo4j(query)


[{'disease': 'ulcer and hangover recovery'},
 {'disease': 'Intestinal wounds'},
 {'disease': 'Juice of leaves applied on wounds'},
 {'disease': 'Ulcer'},
 {'disease': 'BronchitisLithontripticStomach ulcers'},
 {'disease': 'Fresh resin applied on boils and old wounds directly or covered with butter or boiled with rye shoots'},
 {'disease': 'the wound is healed'},
 {'disease': 'to heal wounds or injuries'},
 {'disease': 'gastric ulcer'},
 {'disease': 'Skin inflammation and ulcers'},
 {'disease': 'Stomach ulcers and sciatica'},
 {'disease': 'Fresh leaves applied on wounds'},
 {'disease': 'Externally applied on wounds'},
 {'disease': 'wound healer'},
 {'disease': 'ulcers'},
 {'disease': 'externally applied on wounds'},
 {'disease': 'Wound'},
 {'disease': 'Crushed and topically applied on wounds: haemostatic'},
 {'disease': 'Skin wounds Nose bleeding Internal bleeding'},
 {'disease': 'wounds and haematoma'}]

## 4) Which nodes are USED_FOR wound-related diseases?

In [24]:

# TITLE: Source labels for wound-related USED_FOR links
import pandas as pd

query = """
MATCH (x)-[:USED_FOR]->(d:Disease)
WHERE toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS $kw1
   OR toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS $kw2
   OR toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS $kw3
RETURN labels(x) AS source_labels,
       count(*) AS n
ORDER BY n DESC
"""

params = {
    "kw1": "wound",
    "kw2": "burn",
    "kw3": "ulcer"
}

df = pd.DataFrame(query_neo4j(query, params))

print("Source node labels linked to wound-related diseases:")
df


Source node labels linked to wound-related diseases:


,source_labels,n
0,[Taxon],1503


## 5) List wound-healing taxa (Taxon)

### Wound-healing plants mentioned in articles

In [5]:
import pandas as pd

query = """
MATCH (a:Article)<-[:PART_OF]-(tb:Table)
      -[:MENTIONS_TAXON]->(t:Taxon)
      -[:USED_FOR]->(d:Disease)

WHERE
toLower(coalesce(d.diseaseName,d.name,'')) CONTAINS 'wound'
OR toLower(coalesce(d.diseaseName,d.name,'')) CONTAINS 'ulcer'
OR toLower(coalesce(d.diseaseName,d.name,'')) CONTAINS 'burn'

RETURN
a.title AS article,
t.taxonName AS plant,
coalesce(d.diseaseName, d.name) AS disease
LIMIT 200
"""
pd.DataFrame(query_neo4j(query))

,article,plant,disease
0,Resilience at the border: traditional botanica...,CAME 26284,Crushed and topically applied on wounds: haemo...
1,Diversity and use of ethno-medicinal plants in...,Plantago lanceolata,Crushed and topically applied on wounds: haemo...
2,Ethno-medicinal survey of important plants pra...,Plantago lanceolata,Crushed and topically applied on wounds: haemo...
3,Exploring the power of data mining for uncover...,Plantago lanceolata,Crushed and topically applied on wounds: haemo...
4,Use and Conservation of Medicinal Plants by In...,Plantago lanceolata,Crushed and topically applied on wounds: haemo...
...,...,...,...
195,First large-scale ethnobotanical survey in the...,Chromolaena odorata,burns
196,Astonishing diversity—the medicinal plant mark...,Chromolaena odorata,burns
197,Ethnobotanical study of medicinal plants used ...,Chromolaena odorata,burns
198,Ethnobotanical study of medicinal plants used ...,Chromolaena odorata,burns


### Include IPNI identifier for stable linking

In [38]:
import pandas as pd

query = """
MATCH (a:Article)<-[:PART_OF]-(tb:Table)
      -[:MENTIONS_TAXON]->(t:Taxon)
      -[:USED_FOR]->(d:Disease)

WHERE toLower(coalesce(d.diseaseName,d.name,'')) CONTAINS 'wound'

RETURN
a.title AS article,
t.taxonName AS plant,
t.ipni AS ipni,
coalesce(d.diseaseName, d.name) AS disease
LIMIT 200
"""
pd.DataFrame(query_neo4j(query))

,article,plant,ipni,disease
0,Medicinal Plants Used by Traditional Healers i...,Tamarix aphylla,828051-1,which is applied externally on animal skin to ...
1,Medicinal Plants Used by Traditional Healers i...,Phyllanthus niruri,194900-2,Wound
2,Ethnobotanical study on medicinal plants in Me...,Azadirachta indica,1213180-2,Wounds Nasal infection Earache Scabies Intesti...
3,Ethnobotanical study on medicinal plants in Me...,Azadirachta indica,1213180-2,Anti-bacterial for wounds
4,Ethnobotanical study on medicinal plants in Me...,Balanites aegyptiaca,813589-1,Malaria and Wound
...,...,...,...,...
195,Resilience at the border: traditional botanica...,Plantago lanceolata,321285-2,Crushed and topically applied on wounds: haemo...
196,Resilience at the border: traditional botanica...,CAME 26294,NaN,Decoction: cicatrizing on wounds
197,Resilience at the border: traditional botanica...,Cirsium arvense,195034-1,and to heal wounds
198,Resilience at the border: traditional botanica...,Phaseolus vulgaris,514191-1,Wounds


## 6) Explore evidence fields on USED_FOR relationships (mode, usageType, referenceText, province, etc.)

In [6]:

# TITLE: What properties exist on USED_FOR relationships?
query = """
MATCH (t:Taxon)-[r:USED_FOR]->(d:Disease)
WHERE toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS 'wound'
RETURN
t.name AS taxon,
coalesce(d.diseaseName, d.name) AS disease,
r.usageType AS usageType,
r.mode AS mode,
r.province AS province,
r.referenceText AS referenceText
LIMIT 100
"""
query_neo4j(query)

[{'taxon': None,
  'disease': 'Crushed and topically applied on wounds: haemostatic',
  'usageType': None,
  'mode': None,
  'province': None,
  'referenceText': None},
 {'taxon': None,
  'disease': 'Crushed and topically applied on wounds: haemostatic',
  'usageType': None,
  'mode': None,
  'province': None,
  'referenceText': None},
 {'taxon': None,
  'disease': 'Crushed and topically applied on wounds: haemostatic',
  'usageType': None,
  'mode': None,
  'province': None,
  'referenceText': None},
 {'taxon': None,
  'disease': 'Crushed and topically applied on wounds: haemostatic',
  'usageType': None,
  'mode': None,
  'province': None,
  'referenceText': None},
 {'taxon': None,
  'disease': 'Fresh resin applied on boils and old wounds directly or covered with butter or boiled with rye shoots',
  'usageType': None,
  'mode': None,
  'province': None,
  'referenceText': None},
 {'taxon': None,
  'disease': 'wounds',
  'usageType': None,
  'mode': None,
  'province': None,
  'refere

## 7) Extract structured evidence fields (if present on relationship)

In [28]:
import pandas as pd

query = """
MATCH (:Taxon)-[r:USED_FOR]->(:Disease)
RETURN keys(r) AS rel_props, count(*) AS n
ORDER BY n DESC
LIMIT 20
"""
pd.DataFrame(query_neo4j(query))

,rel_props,n
0,[],31270


In [29]:
query = """
MATCH (n:Name)
RETURN keys(n) AS name_props, count(*) AS n
ORDER BY n DESC
LIMIT 20
"""
pd.DataFrame(query_neo4j(query))

,name_props,n
0,"[rank, name, uri]",903815
1,"[uri, rank, name]",619194
2,"[name, rank, uri]",139355
3,"[uri, name, rank]",123867


In [7]:
import pandas as pd

query = """
MATCH (t:Taxon)-[:USED_FOR]->(d:Disease)
WHERE toLower(coalesce(d.diseaseName, d.name,'')) CONTAINS $kw

OPTIONAL MATCH (t)-[:HAS_NAME]->(n:Name)

RETURN
coalesce(n.name, t.name, '') AS taxon_name,
coalesce(d.diseaseName, d.name) AS disease_text

LIMIT $limit
"""

params = {"kw": "wound", "limit": 200}
pd.DataFrame(query_neo4j(query, params))

,taxon_name,disease_text
0,,Crushed and topically applied on wounds: haemo...
1,,Crushed and topically applied on wounds: haemo...
2,,Crushed and topically applied on wounds: haemo...
3,,Crushed and topically applied on wounds: haemo...
4,,Fresh resin applied on boils and old wounds di...
...,...,...
195,,Wounds
196,,Wounds
197,,Wounds
198,,Wounds


In [8]:
import pandas as pd

query = """
MATCH (t:Taxon)-[:HAS_COMPOUND]->(c)
RETURN labels(c) AS compound_labels, count(*) AS n
ORDER BY n DESC
LIMIT 30
"""
pd.DataFrame(query_neo4j(query))

""


In [9]:
import pandas as pd

query = """
MATCH (t:Taxon)-[:HAS_COMPOUND]->(c)
RETURN labels(c) AS compound_labels, count(*) AS n
ORDER BY n DESC
LIMIT 30
"""
df = pd.DataFrame(query_neo4j(query))
print("Rows returned:", len(df))
df

Rows returned: 0


""


## 8) Mediterranean filter 

In [11]:
query ="""WITH 
[
"Skin cuts and burns",
"to heal wounds",
"Skin burns and wound healing",
"especially for wounds",
"Wounds and cuts",
"skin burns",
"burns",
"Wounds healing and Backache",
"Wounds",
"and burns",
"Skin burn",
"Injury/Fresh wound",
"wounds and haematoma",
"External wounds",
"skin wounds",
"Cut and wound",
"Burns",
"Wound",
"the wound is healed",
"fire burn",
"Skin lesions",
"wound healing"
] AS woundList,

[
'spain','france','italy','greece','turkey','cyprus','lebanon',
'israel','egypt','tunisia','algeria','morocco','libya',
'croatia','slovenia','albania','bosnia','montenegro','malta'
] AS medCountries

MATCH (a:Article)<-[:PART_OF]-(tb:Table)
      -[:MENTIONS_TAXON]->(t:Taxon)
      -[:USED_FOR]->(d:Disease)

WHERE 
  ANY(w IN woundList 
      WHERE toLower(coalesce(d.diseaseName,d.name,'')) = toLower(w))
  AND ANY(country IN medCountries
      WHERE toLower(a.title) CONTAINS country)

RETURN
  a.title AS article,
  t.taxonName AS plant,
  t.ipni AS ipni,
  coalesce(d.diseaseName, d.name) AS disease
LIMIT 200"""
pd.DataFrame(query_neo4j(query))

,article,plant,ipni,disease
0,Wild Plants Used as Herbs and Spices in Italy:...,Olea europaea,610675-1,Wound
1,Wild Plants Used as Herbs and Spices in Italy:...,Tanacetum vulgare,252568-1,skin lesions
2,Wild Plants Used as Herbs and Spices in Italy:...,Anethum graveolens,837530-1,burns
3,Wild Plants Used as Herbs and Spices in Italy:...,Salvia officinalis,456833-1,Wound
4,Wild Plants Used as Herbs and Spices in Italy:...,Brassica nigra,279422-1,Wound
...,...,...,...,...
195,Comparative Medical Ethnobotany of the Senegal...,Allium sativum,528796-1,Wounds
196,Comparative Medical Ethnobotany of the Senegal...,Guiera senegalensis,20011770-1,Wounds
197,Comparative Medical Ethnobotany of the Senegal...,Elaeis guineensis,666802-1,Wound
198,Comparative Medical Ethnobotany of the Senegal...,Manihot esculenta,351790-1,Wound


## 9) Compounds that have a role in would healing 

In [12]:
query =""" MATCH (c:DbChEBI)
WHERE toLower(coalesce(c.definition,'')) CONTAINS "wound"
RETURN c.ascii_name, c.definition
LIMIT 50"""
pd.DataFrame(query_neo4j(query))

,c.ascii_name,c.definition
0,nitrofurazone,A semicarbazone resulting from the formal cond...
1,piperonylic acid,A member of the class of benzodioxoles that is...
2,grandiflorenic acid,A tetracyclic diterpenoid with formula C<small...
3,traumatic acid,A monounsaturated straight-chain dicarboxylic ...
4,asiaticoside,A triterpenoid saponin that is a trisaccharide...
5,"3,6-diaminoacridine",An aminoacridine that is acridine that is subs...
6,N(2)-([biphenyl]-4-ylsulfonyl)-N-hydroxy-N(2)-...,A hydroxamic acid that is <em>N</em>-hydroxy-<...
7,vulnerary,A drug used in treating and healing of wounds.
